# Notebook 4 — Customer & Revenue Analytics

### 📺 Brought to you by **Christian Martinez: AI for Finance**
👉 **Subscribe here: https://www.youtube.com/@christianmartinezAIforFinance**

> *Python for CFOs — a 7-part journey from spreadsheet to strategy.*

---
**Level:** 🟡 Intermediate  
Where does revenue really come from — and where does it leak? Analyze 500 customers: segment profitability, churn drivers, LTV, and a clean AR aging report.

---

## Datasets: `customers.csv` and `ar_aging.csv` (both included)

In [ ]:
import pandas as pd, numpy as np
import matplotlib.pyplot as plt

try:
    cust = pd.read_csv("customers.csv", parse_dates=["signup_date"])
except FileNotFoundError:
    np.random.seed(42); n=500
    cust=pd.DataFrame({
      "customer_id":range(1001,1001+n),
      "region":np.random.choice(["North","South","East","West"],n),
      "segment":np.random.choice(["Enterprise","Mid-Market","SMB"],n,p=[.2,.3,.5]),
      "signup_date":pd.to_datetime("2022-01-01")+pd.to_timedelta(np.random.randint(0,1000,n),"D"),
      "monthly_revenue":np.round(np.random.gamma(2,800,n),2),
      "churned":np.random.choice([0,1],n,p=[.78,.22]),
      "support_tickets":np.random.poisson(3,n),
      "contract_months":np.random.choice([12,24,36],n)})
cust.head()

### 1. Revenue by segment — the 80/20 view

In [ ]:
seg = cust.groupby("segment").agg(
    customers=("customer_id","count"),
    total_monthly_rev=("monthly_revenue","sum"),
    avg_rev=("monthly_revenue","mean")
).round(0).sort_values("total_monthly_rev", ascending=False)
seg["rev_share"]=(seg.total_monthly_rev/seg.total_monthly_rev.sum()).round(3)
seg

### 2. Churn analysis — who's leaving and why

In [ ]:
churn_rate = cust["churned"].mean()
print(f"Overall churn rate: {churn_rate:.1%}\n")

by_seg = cust.groupby("segment")["churned"].mean().round(3)
print("Churn by segment:"); print(by_seg)

# Do unhappy customers (more tickets) churn more?
print("\nAvg support tickets:")
print(cust.groupby("churned")["support_tickets"].mean().round(2))

### 3. Customer Lifetime Value (LTV)
Simple LTV = monthly revenue × expected lifetime (months).

In [ ]:
# Approx expected lifetime from churn: 1 / monthly churn probability
monthly_churn = cust["churned"].mean()/12
cust["expected_lifetime_m"] = 1/monthly_churn
cust["ltv"] = cust["monthly_revenue"] * cust["expected_lifetime_m"]

print(f"Average LTV: ${cust['ltv'].mean():,.0f}")
print("\nLTV by segment:")
print(cust.groupby("segment")["ltv"].mean().round(0))

### 4. Visualize segment revenue & regional split

In [ ]:
fig, ax = plt.subplots(1,2, figsize=(13,4.5))
seg["total_monthly_rev"].plot(kind="bar", ax=ax[0], color="#4f81bd")
ax[0].set_title("Monthly Revenue by Segment"); ax[0].set_ylabel("$")
cust.groupby("region")["monthly_revenue"].sum().plot(
    kind="pie", ax=ax[1], autopct="%1.0f%%")
ax[1].set_title("Revenue by Region"); ax[1].set_ylabel("")
plt.tight_layout(); plt.show()

### 5. Accounts Receivable aging report

In [ ]:
try:
    ar = pd.read_csv("ar_aging.csv")
except FileNotFoundError:
    np.random.seed(1)
    ar=pd.DataFrame({"invoice_id":[f"INV-{i:04d}" for i in range(1,201)],
      "amount":np.round(np.random.gamma(3,4000,200),2),
      "days_outstanding":np.random.choice([15,30,45,60,90,120],200,p=[.3,.25,.18,.12,.1,.05])})

def bucket(d):
    if d<=30: return "0-30"
    if d<=60: return "31-60"
    if d<=90: return "61-90"
    return "90+"
ar["bucket"]=ar["days_outstanding"].apply(bucket)
aging = ar.groupby("bucket")["amount"].sum().reindex(["0-30","31-60","61-90","90+"])
print("AR Aging Summary:"); print(aging.round(0))
print(f"\n⚠️  At-risk (90+ days): ${aging['90+']:,.0f}")
aging.plot(kind="bar", color=["#5cb85c","#f0ad4e","#e8851a","#d9534f"], figsize=(8,4))
plt.title("Accounts Receivable Aging"); plt.ylabel("$"); plt.tight_layout(); plt.show()

### 🎯 Your turn
Which region has the worst churn? Flag every customer with `ltv` below the median **and** 5+ support tickets — that's your at-risk list.

---
### ✅ Recap
- `groupby` powers segment/cohort analysis • churn × tickets reveals drivers • LTV ties retention to dollars • aging buckets surface collection risk
- **Next — Notebook 5:** forecasting the future.

📺 **Subscribe: https://www.youtube.com/@christianmartinezAIforFinance**